# Good vs Anomalous Descriptor Comparison

This notebook compares descriptor behavior between normal and anomalous samples for the same category.

The goal is simple:

- normals should look calm and structured
- anomalies should show stronger or more concentrated evidence
- cross-modal maps should become more informative, not just brighter everywhere


In [ ]:
from pathlib import Path
import csv
import json

try:
    from PIL import Image, ImageOps, ImageDraw
    PIL_AVAILABLE = True
except ImportError:
    Image = None
    ImageOps = None
    ImageDraw = None
    PIL_AVAILABLE = False

try:
    from IPython.display import display
except ImportError:
    display = None

print('PIL available:', PIL_AVAILABLE)

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'raw' / 'MulSen_AD').exists() and (candidate / 'src' / 'notebooks').exists():
            return candidate
    raise RuntimeError(f'Could not locate project root from {start}')


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'data'
INDEX_CSV = DATA_ROOT / 'processed' / 'manifests' / 'dataset_index.csv'
DESCRIPTOR_ROOT = DATA_ROOT / 'processed' / 'descriptors'

assert INDEX_CSV.exists(), f'Missing dataset index: {INDEX_CSV}'
assert DESCRIPTOR_ROOT.exists(), f'Missing descriptor root: {DESCRIPTOR_ROOT}'

print('Project root:', PROJECT_ROOT)
print('Index:', INDEX_CSV)
print('Descriptor root:', DESCRIPTOR_ROOT)

In [ ]:
rows = list(csv.DictReader(INDEX_CSV.open(newline='', encoding='utf-8')))
print('Indexed rows:', len(rows))
print('Categories:', sorted({row['category'] for row in rows}))

## Sample Selection

Choose a category and one defect label to compare against a normal `good` sample.

In [ ]:
CATEGORY = 'button_cell'
DEFECT_LABEL = 'hole'

category_rows = [row for row in rows if row['category'] == CATEGORY]
good_rows = [row for row in category_rows if row['split'] == 'test' and row['defect_label'] == 'good']
anomaly_rows = [row for row in category_rows if row['split'] == 'test' and row['defect_label'] == DEFECT_LABEL]

print('Good test samples:', len(good_rows))
print(f'{DEFECT_LABEL} test samples:', len(anomaly_rows))

good_row = good_rows[0] if good_rows else None
anomaly_row = anomaly_rows[0] if anomaly_rows else None
print('Selected good sample:', None if good_row is None else good_row['sample_id'])
print('Selected anomalous sample:', None if anomaly_row is None else anomaly_row['sample_id'])

In [ ]:
def project_path(path_like):
    path = Path(path_like)
    return path if path.is_absolute() else PROJECT_ROOT / path


def descriptor_manifest_path(descriptor_type, row):
    return (
        DESCRIPTOR_ROOT
        / descriptor_type
        / row['category']
        / row['split']
        / row['defect_label']
        / row['sample_id']
        / 'manifest.json'
    )


def load_manifest(path):
    return json.loads(project_path(path).read_text(encoding='utf-8'))


def load_global_json(path):
    return json.loads(project_path(path).read_text(encoding='utf-8'))


def artifact_path(manifest, name):
    for artifact in manifest:
        if artifact['name'] == name:
            return project_path(artifact['path'])
    raise KeyError(name)


In [ ]:
required_types = ['infrared', 'pointcloud', 'crossmodal']

def check_manifests(row):
    status = {}
    for descriptor_type in required_types:
        path = descriptor_manifest_path(descriptor_type, row)
        status[descriptor_type] = path.exists()
    return status


if good_row is not None:
    print('Good descriptor availability:', check_manifests(good_row))
if anomaly_row is not None:
    print('Anomalous descriptor availability:', check_manifests(anomaly_row))


If a descriptor manifest is missing for the anomalous sample, generate it first with the descriptor scripts before continuing.

In [ ]:
if good_row is None or anomaly_row is None:
    raise RuntimeError('Could not find both good and anomalous rows for the chosen category/defect.')

good_crossmodal_manifest = descriptor_manifest_path('crossmodal', good_row)
anomaly_crossmodal_manifest = descriptor_manifest_path('crossmodal', anomaly_row)

if not good_crossmodal_manifest.exists() or not anomaly_crossmodal_manifest.exists():
    print('At least one crossmodal manifest is missing.')
    print('Generate more descriptors before continuing, for example:')
    print(f"python3 scripts/run_data_pipeline.py --categories {CATEGORY}")
    raise RuntimeError('Missing descriptor manifests for comparison.')

## Numeric Comparison

In [ ]:
def collect_global_metrics(row):
    metrics = {}
    for descriptor_type in required_types:
        manifest = load_manifest(descriptor_manifest_path(descriptor_type, row))
        global_artifact = [artifact for artifact in manifest if artifact['kind'] == 'global'][0]
        metrics[descriptor_type] = load_global_json(global_artifact['path'])
    return metrics


good_metrics = collect_global_metrics(good_row)
anomaly_metrics = collect_global_metrics(anomaly_row)

print('GOOD sample:', good_row['sample_id'])
print(json.dumps(good_metrics, indent=2, sort_keys=True))

print('\nANOMALOUS sample:', anomaly_row['sample_id'])
print(json.dumps(anomaly_metrics, indent=2, sort_keys=True))

In [ ]:
if 'good_metrics' not in globals() or 'anomaly_metrics' not in globals():
    good_metrics = collect_global_metrics(good_row)
    anomaly_metrics = collect_global_metrics(anomaly_row)

print('Key comparison summary:')
print('Infrared hotspot ratio:', good_metrics['infrared'].get('hotspot_ratio'), '->', anomaly_metrics['infrared'].get('hotspot_ratio'))
print('Infrared std intensity:', good_metrics['infrared'].get('std_intensity'), '->', anomaly_metrics['infrared'].get('std_intensity'))
print('Pointcloud coverage ratio:', good_metrics['pointcloud'].get('coverage_ratio'), '->', anomaly_metrics['pointcloud'].get('coverage_ratio'))
print('Crossmodal agreement mean:', good_metrics['crossmodal'].get('agreement_mean'), '->', anomaly_metrics['crossmodal'].get('agreement_mean'))
print('Crossmodal agreement std:', good_metrics['crossmodal'].get('agreement_std'), '->', anomaly_metrics['crossmodal'].get('agreement_std'))
print('Crossmodal support mean:', good_metrics['crossmodal'].get('support_mean'), '->', anomaly_metrics['crossmodal'].get('support_mean'))

## Visual Comparison

In [ ]:
if not PIL_AVAILABLE:
    raise RuntimeError('Pillow is required for visual comparison.')


def prepare_tile(path, title, size=(280, 190)):
    image = Image.open(project_path(path)).convert('RGB')
    tile = ImageOps.pad(image, size, color=(255, 255, 255))
    canvas = Image.new('RGB', (size[0], size[1] + 28), 'white')
    canvas.paste(tile, (0, 28))
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), title, fill='black')
    return canvas


def contact_sheet(tiles, columns=4):
    tile_width, tile_height = tiles[0].size
    rows_needed = (len(tiles) + columns - 1) // columns
    sheet = Image.new('RGB', (tile_width * columns, tile_height * rows_needed), 'lightgray')
    for idx, tile in enumerate(tiles):
        x = (idx % columns) * tile_width
        y = (idx // columns) * tile_height
        sheet.paste(tile, (x, y))
    return sheet


def sample_tiles(row):
    tiles = [
        prepare_tile(row['rgb_path'], f"RGB {row['defect_label']}"),
        prepare_tile(row['ir_path'], f"IR {row['defect_label']}"),
    ]
    ir_manifest = load_manifest(descriptor_manifest_path('infrared', row))
    pc_manifest = load_manifest(descriptor_manifest_path('pointcloud', row))
    cm_manifest = load_manifest(descriptor_manifest_path('crossmodal', row))

    wanted = [
        ('infrared', ir_manifest, ['infrared_normalized', 'infrared_hotspot']),
        ('pointcloud', pc_manifest, ['pointcloud_depth_topdown', 'pointcloud_density_topdown']),
        ('crossmodal', cm_manifest, ['crossmodal_geometric_support', 'crossmodal_agreement']),
    ]

    for _, manifest, names in wanted:
        for name in names:
            tiles.append(prepare_tile(artifact_path(manifest, name), name))
    return tiles


In [ ]:
good_sheet = contact_sheet(sample_tiles(good_row), columns=4)
anomaly_sheet = contact_sheet(sample_tiles(anomaly_row), columns=4)

print('GOOD sample visual comparison')
if display is not None:
    display(good_sheet)
else:
    good_sheet

print('ANOMALOUS sample visual comparison')
if display is not None:
    display(anomaly_sheet)
else:
    anomaly_sheet

## How To Judge The Result

Useful signs:

- anomalous hotspot maps become more localized or stronger than `good`
- point-cloud depth or density changes in defect-supporting regions
- crossmodal agreement is more selective than raw support
- anomalous samples show more informative agreement/support statistics than normal samples

Bad signs:

- all maps look nearly identical between normal and anomalous samples
- agreement maps are always almost blank or always saturated
- point-cloud projections are too sparse to support localization
- numeric summaries move only randomly instead of consistently
